In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision

from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

from sklearn.metrics import classification_report, roc_auc_score, f1_score

import numpy as np
from tqdm import tqdm

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
DATA_DIR = r"C:\CliniScan\classification_data"

TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR = os.path.join(DATA_DIR, "val")

In [4]:
import torchvision.transforms as transforms

transform_train = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

transform_val = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

In [5]:
train_dataset = ImageFolder(TRAIN_DIR, transform=transform_train)
val_dataset = ImageFolder(VAL_DIR, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)

Classes: ['Abnormal', 'Normal']


In [6]:
model = torchvision.models.densenet121(
    weights=torchvision.models.DenseNet121_Weights.DEFAULT
)

model.classifier = nn.Linear(
    model.classifier.in_features,
    num_classes
)

model = model.to(device)

In [7]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.0003)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=5,
    gamma=0.1
)

EPOCHS = 10

best_val_acc = 0

In [8]:
torch.save(model.state_dict(), "classifier_densenet121.pth")

In [ ]:
all_labels = []
all_preds = []

model.eval()

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs,1)

        all_labels.extend(labels.numpy())
        all_preds.extend(preds.cpu().numpy())

print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names
))-

              precision    recall  f1-score   support

    Abnormal       0.29      0.17      0.21       836
      Normal       0.70      0.82      0.76      1990

    accuracy                           0.63      2826
   macro avg       0.50      0.50      0.49      2826
weighted avg       0.58      0.63      0.60      2826

